# SchemaShift on Colab

Runs an open model on the SchemaShift tasks using a free Colab GPU, with the same
`scripts/run_models.py` used on the laptop.

1. **Runtime -> Change runtime type -> T4 GPU.**
2. Run the cells in order. Step 2 asks you to upload `schemashift_student_club.zip`
   (scripts + the four changed databases + `tasks.jsonl`).
3. The last cell downloads `results.zip`. Copy its files into the project's `results/` folder.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project zip and unpack it

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # choose schemashift_student_club.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("/content/Text_To_Sql")
%cd /content/Text_To_Sql
!ls scripts data/schemashift/student_club

### (Optional) Keep results in Google Drive

Colab deletes its files when the session ends. Run this cell to save results straight
into your Drive (`My Drive/SchemaShift/results`), so a disconnect loses nothing and a run
can continue later.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/SchemaShift/results
!rm -rf results && ln -s /content/drive/MyDrive/SchemaShift/results results

## 3. Install the Python package

In [ ]:
!pip -q install sqlglot==30.23.0

## 4. Install Ollama and start it

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)

## 5. Download the model

In [ ]:
MODEL = "qwen2.5-coder:7b"
!ollama pull {MODEL}

## 6. Test on the first 10 tasks

Each line shows `ok`, `wrong` or `ERROR`. The table at the end counts correct answers
per kind of change.

In [ ]:
!python scripts/run_models.py data/schemashift/student_club --model {MODEL} --limit 10

## 7. Full runs (all 144 tasks)

The first run continues from the 10 tasks above. The second adds the one-line hint
(e.g. "Rows with is_deleted = 1 have been deleted and must be ignored").

In [ ]:
!python scripts/run_models.py data/schemashift/student_club --model {MODEL}
!python scripts/run_models.py data/schemashift/student_club --model {MODEL} --hint

## 8. Download the results

In [ ]:
!zip -qr /content/results.zip results/
files.download("/content/results.zip")